# Neural Network Month · Project 3
## Notebook 3 of 3 — Does the Wiring Help?

Here is the real question. A neuron's own measurements already say a lot about its class. **Does adding its connections help?**

We compare three models under identical **stratified 5-fold cross-validation**:

| Model | What it sees |
|---|---|
| **Linear regression** | each neuron's five measurements, nothing else (fit to the one-hot class, pick the highest) |
| **Random forest** | the same five measurements, with a flexible non-linear model |
| **GCN** | the same five measurements **plus the wiring** |

The two baselines see no graph at all, so any gap between them and the GCN is down to the connections. We score all three on five measures and draw them on one radar plot. Four figures.

**One note on R².** R² is a regression score, and this is a classification task. We compute it on the models' predicted **class probabilities** against the true one-hot answer (essentially a rescaled Brier score), so it rewards confident, well-calibrated predictions. For linear regression we use its raw outputs as the scores.

## 1 · Setup, data, code

In [ ]:
import os, sys, glob, zipfile, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import FancyBboxPatch, Patch
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

# One palette for the series: steel blue and lime green lead, like a lit-up brain scan.
P = {
    "steel":  "#4682B4",  # steel blue
    "lime":   "#8CC63F",  # lime green
    "teal":   "#2FB5A6",
    "violet": "#7A6BC4",
    "berry":  "#C2417E",
    "ink":    "#1E2B3A",  # text
    "paper":  "#F2F6F9",  # background
    "grid":   "#D3DCE4",
    "stone":  "#8896A5",  # neutral
    "rust":   "#B5452D",  # warnings (proxy graph)
}
# The smell pathway, nose to brain to body: lime -> teal -> steel -> violet -> berry
CLASSES = ["ALRN", "ALLN", "ALPN", "TOON", "DN"]
CLASS_NAMES = {"ALRN": "Receptor", "ALLN": "Local", "ALPN": "Projection", "TOON": "Third-Order", "DN": "Descending"}
CLASS_COLORS = {"ALRN": P["lime"], "ALLN": P["teal"], "ALPN": P["steel"], "TOON": P["violet"], "DN": P["berry"]}
MODEL_COLORS = {"Linear Regression": P["stone"], "Random Forest": P["steel"], "GCN": P["lime"]}
SEQ_CMAP = LinearSegmentedColormap.from_list("seq", [P["paper"], P["steel"], P["ink"]])

plt.rcParams.update({
    "figure.facecolor": P["paper"], "axes.facecolor": P["paper"],
    "axes.edgecolor": P["ink"], "axes.labelcolor": P["ink"],
    "text.color": P["ink"], "xtick.color": P["ink"], "ytick.color": P["ink"],
    "axes.grid": True, "grid.color": P["grid"], "grid.alpha": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 13, "axes.titleweight": "bold",
    "figure.titlesize": 16, "figure.titleweight": "bold",
    "font.size": 11, "figure.dpi": 110,
})

Path("figures").mkdir(exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f"figures/{name}.png", dpi=200, bbox_inches="tight", facecolor=fig.get_facecolor())

def proxy_note(fig, y=-0.03):
    """Stamp figures that depend on a stand-in graph, so nobody mistakes them for real wiring."""
    if globals().get("EDGE_SOURCE", "") == "proxy":
        fig.text(0.5, y, "PROXY GRAPH: neighbours chosen by feature similarity, not real synapses",
                 ha="center", color=P["rust"], fontsize=10, fontweight="bold")

In [ ]:
# Where is the data? Download archive.zip from the Kaggle page and upload it to /content
# (the Colab file panel), or point ZIP_PATH at it. The cell also tries a Kaggle download.
ZIP_PATH = "/content/archive.zip"
DATA_DIR = "fly_data"

def locate_root(base):
    for p in glob.glob(f"{base}/**/S1_hemibrain_neuron_layers.csv", recursive=True):
        return os.path.dirname(p)
    return None

root = locate_root(DATA_DIR) if os.path.isdir(DATA_DIR) else None
if root is None and os.path.exists(ZIP_PATH):
    zipfile.ZipFile(ZIP_PATH).extractall(DATA_DIR)
    root = locate_root(DATA_DIR)
if root is None:
    try:
        import kagglehub
        root = locate_root(kagglehub.dataset_download("thedevastator/hemibrain-neuronal-connectome"))
    except Exception as e:
        print("Kaggle download did not work:", e)
if root is None:
    raise FileNotFoundError("Could not find the hemibrain CSVs. Upload the Kaggle zip and set ZIP_PATH.")
print("Data folder:", root)

In [ ]:
# ---- The only places we touch the data ----
N_FOLDS, SEED = 5, 42
FEATURES = ["pre", "post", "downstream", "voxels", "soma"]   # measured properties, no label information

def build_nodes(root):
    files = {"ALRN": "S3_hemibrain_ALRN_meta.csv", "ALLN": "S4_hemibrain_ALLN_meta.csv",
             "ALPN": "S5_hemibrain_ALPN_meta.csv", "TOON": "S6_hemibrain_TOON_meta.csv",
             "DN": "S7_hemibrain_DN_meta.csv"}
    cols = ["bodyid", "status"] + FEATURES
    nodes = pd.concat([pd.read_csv(os.path.join(root, f), usecols=cols).assign(cls=c) for c, f in files.items()],
                      ignore_index=True)
    # A neuron listed in two class tables has no single ground-truth label, so it is set aside.
    ambiguous = nodes.bodyid[nodes.bodyid.duplicated(keep=False)].unique()
    nodes = nodes[~nodes.bodyid.isin(ambiguous)].reset_index(drop=True)
    nodes["soma"] = nodes["soma"].astype(int)
    nodes["label"] = nodes.cls.map(CLASSES.index)
    # The authors' layer score is only used to DRAW the pathway, never as a feature or a target.
    layers = pd.read_csv(os.path.join(root, "S1_hemibrain_neuron_layers.csv")).rename(columns={"bodyId": "bodyid"})
    nodes = nodes.merge(layers[["bodyid", "layer_mean"]], on="bodyid", how="left")
    nodes["fold"] = -1
    for k, (_, te) in enumerate(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(nodes, nodes.label)):
        nodes.loc[te, "fold"] = k
    return nodes, len(ambiguous)

In [ ]:
# ---- Where the wiring comes from ----
# Main route: the neuPrint server at Janelia (needs a free account token).
# Backup route: a clearly labelled PROXY graph built from the node table alone.
MIN_SYNAPSES      = 3                    # ignore connections weaker than this (a common guard against false synapse calls)
NEUPRINT_SERVER   = "neuprint.janelia.org"
NEUPRINT_DATASETS = ["hemibrain:v1.2.1", "hemibrain:v1.2", "hemibrain:v1.1"]   # the one that matches our IDs best is used
EDGES_CSV         = None                 # optional: path to your own edge file with columns bodyId_pre, bodyId_post, weight
USE_PROXY_IF_NEEDED = True

def get_token():
    """Looks for NEUPRINT_TOKEN in environment variables or Colab Secrets; in Colab, offers a hidden prompt."""
    tok = os.environ.get("NEUPRINT_TOKEN")
    if tok: return tok
    try:
        from google.colab import userdata
        tok = userdata.get("NEUPRINT_TOKEN")
        if tok: return tok
    except Exception:
        pass
    if "google.colab" in sys.modules:
        import getpass
        return getpass.getpass("neuPrint token (press Enter to skip and use the proxy graph): ").strip() or None
    return None

def fetch_edges_neuprint(ids, token):
    try:
        import neuprint
    except ImportError:
        get_ipython().run_line_magic("pip", "install -q neuprint-python")
        import neuprint
    from neuprint import Client, fetch_adjacencies
    ids = [int(i) for i in ids]
    sample = [int(i) for i in np.random.default_rng(0).choice(ids, size=min(1000, len(ids)), replace=False)]
    best = None
    for ds in NEUPRINT_DATASETS:                      # the supplement and the server can differ by a data version
        try:
            client = Client(NEUPRINT_SERVER, dataset=ds, token=token)
            found = int(client.fetch_custom(f"MATCH (n:Neuron) WHERE n.bodyId IN {sample} RETURN count(n) AS n")["n"].iloc[0])
        except Exception as e:
            print(f"  {ds}: could not be queried ({str(e)[:90]})"); continue
        print(f"  {ds}: {found} of {len(sample)} sampled neuron IDs found")
        if best is None or found > best[1]: best = (ds, found, client)
    if best is None: raise RuntimeError("no neuPrint dataset could be queried")
    ds, found, client = best
    print(f"Using {ds}. Fetching connections among {len(ids)} neurons (this can take a few minutes)...")
    _, conn = fetch_adjacencies(sources=ids, targets=ids, min_total_weight=MIN_SYNAPSES,
                                omit_rois=True, weight_props=["weight"], client=client)
    return conn[["bodyId_pre", "bodyId_post", "weight"]], f"neuPrint {ds}"

def proxy_edges(nodes, k=10):
    """Backup graph: link each neuron to its k most similar neurons by measured features. NOT synapses."""
    from sklearn.neighbors import NearestNeighbors
    X = np.log1p(nodes[["pre", "post", "downstream", "voxels"]].astype(float))
    X["soma"] = nodes["soma"].astype(float)
    X = ((X - X.mean()) / X.std()).values
    nn = NearestNeighbors(n_neighbors=k + 1, metric="cosine").fit(X)
    _, idx = nn.kneighbors(X)
    ids = nodes.bodyid.values
    rows = [(ids[i], ids[j], 1.0) for i in range(len(ids)) for j in idx[i, 1:]]
    return pd.DataFrame(rows, columns=["bodyId_pre", "bodyId_post", "weight"])

def get_edges(nodes):
    edges, source = None, None
    if EDGES_CSV:
        edges, source = pd.read_csv(EDGES_CSV), f"file {EDGES_CSV}"
    else:
        token = get_token()
        if token:
            try:
                edges, source = fetch_edges_neuprint(nodes.bodyid.values, token)
            except Exception as e:
                print("neuPrint route failed:", repr(e)[:300])
        else:
            print("No neuPrint token found.")
    if edges is None:
        if not USE_PROXY_IF_NEEDED: raise RuntimeError("No edges available.")
        print("\n*** Falling back to the PROXY graph: neighbours are chosen by feature similarity, NOT real synapses. ***\n")
        return proxy_edges(nodes), "proxy"
    # Necessary tidying: keep our neurons only, drop self-connections, merge repeated pairs, apply the weight floor
    keep = set(nodes.bodyid)
    edges = edges[edges.bodyId_pre.isin(keep) & edges.bodyId_post.isin(keep) & (edges.bodyId_pre != edges.bodyId_post)]
    edges = edges.groupby(["bodyId_pre", "bodyId_post"], as_index=False)["weight"].sum()
    return edges[edges.weight >= MIN_SYNAPSES].reset_index(drop=True), source

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import scipy.sparse as sp
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

def make_features(nodes):
    """Log-scale the heavy-tailed counts, then standardise. Uses no labels."""
    X = np.log1p(nodes[["pre", "post", "downstream", "voxels"]].astype(float).clip(lower=0))
    X["soma"] = nodes["soma"].astype(float)
    return ((X - X.mean()) / X.std()).values.astype(np.float32)

def build_adjacency(edges, id_to_idx, n):
    """Normalised adjacency A_hat. Averaging over neighbours = multiplying by this matrix."""
    src = edges.bodyId_pre.map(id_to_idx).values
    dst = edges.bodyId_post.map(id_to_idx).values
    w = np.log1p(edges.weight.values)                       # tame the heavy-tailed synapse counts
    A = sp.coo_matrix((w, (src, dst)), shape=(n, n)).tocsr()
    A = A + A.T                                             # synapses are directed; the GCN listens both ways
    A = A + sp.eye(n)                                       # every neuron also keeps its own signal
    dinv = sp.diags(np.asarray(A.sum(1)).ravel() ** -0.5)
    A_hat = (dinv @ A @ dinv).tocoo()
    idx = torch.tensor(np.vstack([A_hat.row, A_hat.col]), dtype=torch.long)
    return torch.sparse_coo_tensor(idx, torch.tensor(A_hat.data, dtype=torch.float32), (n, n)).coalesce()

class GCN(nn.Module):
    """Two graph-convolution layers: each neuron averages its neighbours' signals, then a small network transforms them."""
    def __init__(self, n_in, n_hidden, n_out, dropout=0.5):
        super().__init__()
        self.lin1, self.lin2, self.drop = nn.Linear(n_in, n_hidden), nn.Linear(n_hidden, n_out), nn.Dropout(dropout)
    def hidden(self, X, A):                                 # layer 1: gather direct partners
        return F.relu(torch.sparse.mm(A, self.lin1(self.drop(X))))
    def forward(self, X, A):                                # layer 2: gather partners of partners
        return torch.sparse.mm(A, self.lin2(self.drop(self.hidden(X, A))))

def train_gcn(X, A, y, tr_idx, va_idx, n_classes, epochs=200, lr=0.02, wd=5e-4, hidden=32, seed=0):
    torch.manual_seed(seed)
    model = GCN(X.shape[1], hidden, n_classes)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    counts = torch.bincount(y[tr_idx], minlength=n_classes).float()
    cw = (len(tr_idx) / (n_classes * counts.clamp(min=1)))      # rarer classes count more
    hist = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    for _ in range(epochs):
        model.train()
        loss = F.cross_entropy(model(X, A)[tr_idx], y[tr_idx], weight=cw)
        opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            out = model(X, A)
            hist["train_loss"].append(F.cross_entropy(out[tr_idx], y[tr_idx], weight=cw).item())
            hist["val_loss"].append(F.cross_entropy(out[va_idx], y[va_idx], weight=cw).item())
            hist["train_acc"].append((out[tr_idx].argmax(1) == y[tr_idx]).float().mean().item())
            hist["val_acc"].append((out[va_idx].argmax(1) == y[va_idx]).float().mean().item())
    return model, hist

def score_all(y_true, probs, n_classes):
    """Five scores per model. Precision, recall and F1 are macro-averaged (every class counts equally).
    R-squared compares the model's class probabilities with the true one-hot answer."""
    pred = probs.argmax(1)
    Y = np.eye(n_classes)[y_true]
    r2 = 1 - ((Y - probs) ** 2).sum() / ((Y - Y.mean(0)) ** 2).sum()
    return {"Accuracy": accuracy_score(y_true, pred),
            "F1": f1_score(y_true, pred, average="macro", zero_division=0),
            "Precision": precision_score(y_true, pred, average="macro", zero_division=0),
            "Recall": recall_score(y_true, pred, average="macro", zero_division=0),
            "R²": r2}

In [ ]:
if os.path.exists("fly_nodes.csv") and os.path.exists("fly_edges.csv"):      # saved by Notebook 1
    nodes, edges = pd.read_csv("fly_nodes.csv"), pd.read_csv("fly_edges.csv")
    EDGE_SOURCE = open("fly_edge_source.txt").read().strip()
else:                                                                         # otherwise rebuild the same objects
    nodes, _ = build_nodes(root)
    edges, EDGE_SOURCE = get_edges(nodes)
print(f"{len(nodes)} neurons | {len(edges)} connections | edge source: {EDGE_SOURCE}")

In [ ]:
X_np = make_features(nodes)
y_np = nodes.label.values
id_to_idx = {b: i for i, b in enumerate(nodes.bodyid)}
A = build_adjacency(edges, id_to_idx, len(nodes))
X, y = torch.tensor(X_np), torch.tensor(y_np, dtype=torch.long)
n_classes = len(CLASSES)
folds = nodes.fold.values
if EDGE_SOURCE == "proxy":
    print("WARNING: running on the PROXY graph. Results show the pipeline works, not what real wiring adds.")

## 2 · Train and score every fold

Same hyperparameters for every fold, set in advance and **not tuned on the held-out neurons**. The random forest uses balanced class weights, like the GCN's weighted loss.

In [ ]:
MODELS = ["Linear Regression", "Random Forest", "GCN"]
oof = {m: np.zeros((len(nodes), n_classes)) for m in MODELS}      # out-of-fold class scores
rows = []
fold_models = {}
for k in range(N_FOLDS):
    tr, te = np.where(folds != k)[0], np.where(folds == k)[0]
    # 1) linear regression on the one-hot class
    lr = LinearRegression().fit(X_np[tr], np.eye(n_classes)[y_np[tr]])
    oof["Linear Regression"][te] = lr.predict(X_np[te])
    # 2) random forest
    rf = RandomForestClassifier(300, class_weight="balanced_subsample", random_state=0, n_jobs=-1).fit(X_np[tr], y_np[tr])
    oof["Random Forest"][te] = rf.predict_proba(X_np[te])
    # 3) GCN (sees the wiring)
    model, _ = train_gcn(X, A, y, torch.tensor(tr), torch.tensor(te), n_classes, seed=k)
    model.eval()
    with torch.no_grad():
        oof["GCN"][te] = torch.softmax(model(X, A), 1).numpy()[te]
    fold_models[k] = model
    for m in MODELS:
        rows.append({"Model": m, "Fold": k + 1, **score_all(y_np[te], oof[m][te], n_classes)})
    print(f"fold {k+1} done")

scores = pd.DataFrame(rows)
METRICS = ["Accuracy", "F1", "Precision", "Recall", "R²"]
summary = scores.groupby("Model")[METRICS].agg(["mean", "std"]).reindex(MODELS)
display(summary.round(3))

## 3 · Figure 7 — the radar

In [ ]:
angles = np.linspace(0, 2 * np.pi, len(METRICS), endpoint=False).tolist(); angles += angles[:1]
mean = scores.groupby("Model")[METRICS].mean().reindex(MODELS)
fig, ax = plt.subplots(figsize=(7.6, 7.6), subplot_kw=dict(polar=True))
for m in MODELS:
    v = np.clip(mean.loc[m].values, 0, 1).tolist(); v += v[:1]
    ax.plot(angles, v, color=MODEL_COLORS[m], linewidth=3, label=m, zorder=3 if m == "GCN" else 2)
    ax.fill(angles, v, color=MODEL_COLORS[m], alpha=0.2)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(METRICS, fontsize=13, fontweight="bold")
ax.set_ylim(0, 1); ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0]); ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], color=P["stone"], fontsize=9)
ax.set_rlabel_position(22); ax.grid(color=P["grid"]); ax.spines["polar"].set_color(P["grid"]); ax.set_facecolor(P["paper"])
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.06), ncol=3, fontsize=11)
ax.set_title("Model Comparison Radar", pad=26)
fig.tight_layout(); proxy_note(fig, y=0.02); save_fig(fig, "07_model_comparison_radar"); plt.show()
if (mean.values < 0).any(): print("Note: a negative R² was drawn at 0.")

**How to read it.** Bigger polygons are better on every spoke. Accuracy tends to look similar across models because two big classes dominate. The macro **F1, precision and recall** spokes show whether a model also gets the small classes right, and that is where the wiring should matter. The fold-by-fold spread is in the table above; a gap smaller than the spread is not convincing.

## 4 · Figure 8 — where the GCN goes wrong

In [ ]:
pred = oof["GCN"].argmax(1)
cm = confusion_matrix(y_np, pred, labels=range(n_classes))
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(7.2, 6.2))
ax.imshow(cmn, cmap=SEQ_CMAP, vmin=0, vmax=1); ax.grid(False)
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, f"{cm[i, j]}\n{cmn[i, j]:.0%}", ha="center", va="center", fontsize=11, fontweight="bold",
                color="white" if cmn[i, j] > 0.5 else P["ink"])
names = [CLASS_NAMES[c] for c in CLASSES]
ax.set_xticks(range(n_classes)); ax.set_xticklabels(names, rotation=20); ax.set_yticks(range(n_classes)); ax.set_yticklabels(names)
ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
ax.set_title("Out-of-Fold Confusion Matrix")
fig.tight_layout(); proxy_note(fig, y=-0.02); save_fig(fig, "08_confusion_matrix"); plt.show()

Each row is a true class; the diagonal is correct. Off-diagonal cells show which stages the model mixes up, and they should be neighbours on the pathway if the model has learned the structure.

## 5 · Figure 9 — per-class F1

In [ ]:
f1_by = {m: f1_score(y_np, oof[m].argmax(1), average=None, labels=range(n_classes), zero_division=0) for m in MODELS}
fig, ax = plt.subplots(figsize=(11, 4.9))
x = np.arange(n_classes); w = 0.26
for i, m in enumerate(MODELS):
    bars = ax.bar(x + (i - 1) * w, f1_by[m], w, color=MODEL_COLORS[m], label=m)
    ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=8.5)
ax.set_xticks(x); ax.set_xticklabels([CLASS_NAMES[c] for c in CLASSES]); ax.set_ylim(0, 1.1)
ax.set_ylabel("F1 score"); ax.legend(frameon=False, ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.1))
ax.set_title("Per-Class F1 by Model")
fig.tight_layout(); proxy_note(fig, y=-0.12); save_fig(fig, "09_per_class_f1"); plt.show()

This is the fairest comparison. If the GCN beats the baselines mainly on the *small* classes (Local, Projection, Descending), that is evidence the wiring carries information the measurements do not.

## 6 · Figure 10 — before and after the GCN

In [ ]:
from sklearn.manifold import TSNE
te = np.where(folds == 0)[0]
with torch.no_grad():
    H = fold_models[0].eval().hidden(X, A).numpy()[te]               # what the GCN built inside, for held-out neurons
emb = {"Raw Features": X_np[te], "GCN Embedding": H}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
for ax, (title, M) in zip(axes, emb.items()):
    Z = TSNE(n_components=2, init="pca", perplexity=30, random_state=0).fit_transform(M)
    for c in range(n_classes):
        sel = y_np[te] == c
        ax.scatter(Z[sel, 0], Z[sel, 1], s=11, alpha=0.8, color=CLASS_COLORS[CLASSES[c]], label=CLASS_NAMES[CLASSES[c]], linewidths=0)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
axes[1].legend(frameon=False, markerscale=2.4, loc="center left", bbox_to_anchor=(1.0, 0.5))
fig.suptitle("Before and After GCN", y=1.0)
fig.tight_layout(); proxy_note(fig, y=-0.01); save_fig(fig, "10_before_and_after_gcn"); plt.show()

Each dot is a held-out neuron, placed by similarity (t-SNE). **Left:** neurons arranged by their own measurements. **Right:** the same neurons arranged by what the GCN built after gathering their neighbours' signals. If classes separate more cleanly on the right, the graph is adding structure. (t-SNE shows relative closeness only; distances and cluster sizes are not literal.)

## What we showed

- The same five measurements, with and without wiring, scored five ways under identical stratified 5-fold cross-validation.
- Where the wiring helps (the radar, the per-class bars) and where it doesn't.
- A GCN with a few hundred parameters, small enough to explain to a general audience.

**Caveats worth printing:**

- Some class labels were defined using connectivity, so part of the GCN's advantage may be re-finding how the labels were made.
- Scores come from one fly's half-brain and one choice of connection threshold.
- This is an educational demonstration, not a neuroscience finding.

**Coming up:** the final project of the month puts real neuroscience beside everything we have built.

*Data: Schlegel, Bates et al. 2021 (eLife) and Scheffer et al. 2020 (eLife), Janelia FlyEM hemibrain, CC BY 4.0. Please credit the original authors.*

In [ ]:
# Optional: download figures and the score table
try:
    import shutil
    from google.colab import files
    scores.to_csv("model_scores.csv", index=False)
    shutil.make_archive("nb3_figures", "zip", "figures")
    files.download("nb3_figures.zip"); files.download("model_scores.csv")
except ImportError:
    print("Figures are saved in the 'figures' folder.")